# Binary logistic regression from scratch

Predict a binary outcome, plot its decision boundary, and inspect cross-entropy
convergence. All data are generated locally; run after `pip install -e '.[dev]'`.

$P(y=1\mid x)=\sigma(x^T w+b)$, with an unpenalized intercept. See the
[gradient derivation](../docs/logistic_regression.md) for the objective and stability details.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LogisticRegression as SklearnLogisticRegression

from minilearn import LogisticRegression
from minilearn.linear_model.logistic_regression import binary_cross_entropy, sigmoid
from minilearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from minilearn.utils import StandardScaler, train_test_split

rng = np.random.default_rng(42)
X = rng.normal(size=(400, 2))
probability = sigmoid(X @ np.array([1.7, -1.2]) + 0.5)
y = rng.binomial(1, probability)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
scaler = StandardScaler().fit(X_train)
train_scaled = scaler.transform(X_train)
test_scaled = scaler.transform(X_test)

## Fit and compare on exactly the same data

Use `alpha=1` for L2, matching scikit-learn's `C=1/alpha=1`. Only the coefficients
are penalized. Scaling uses the training split; the test split remains held out.
Both models receive the same scaled feature matrix.

In [ ]:
alpha = 1.0
model = LogisticRegression(alpha=alpha, learning_rate=0.5, tol=1e-10, random_state=42).fit(
    train_scaled, y_train
)
reference = SklearnLogisticRegression(C=1 / alpha, tol=1e-12, max_iter=10000).fit(
    train_scaled, y_train
)
assert model.converged_
np.testing.assert_allclose(model.coef_, reference.coef_[0], atol=1e-6, rtol=0)
np.testing.assert_allclose(model.intercept_, reference.intercept_[0], atol=1e-6, rtol=0)
np.testing.assert_allclose(
    model.predict_proba(test_scaled), reference.predict_proba(test_scaled), atol=1e-7, rtol=0
)
np.testing.assert_array_equal(model.predict(test_scaled), reference.predict(test_scaled))
print(f"Converged in {model.n_iter_} updates.")
print(f"Maximum coefficient difference: {np.max(np.abs(model.coef_ - reference.coef_[0])):.3e}")

In [ ]:
print(
    f"{'Model':<14} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Log loss':>10}"
)
for name, estimator in (("MiniLearn", model), ("scikit-learn", reference)):
    predicted = estimator.predict(test_scaled)
    positive_probability = estimator.predict_proba(test_scaled)[:, 1]
    print(
        f"{name:<14} {accuracy_score(y_test, predicted):10.6f} "
        f"{precision_score(y_test, predicted):10.6f} {recall_score(y_test, predicted):10.6f} "
        f"{f1_score(y_test, predicted):10.6f} "
        f"{binary_cross_entropy(y_test, positive_probability):10.6f}"
    )

## Decision boundary and optimization history

Color shows the predicted probability of class 1. The dashed contour is $p=0.5$;
points show held-out observations. The loss includes the L2 penalty and the
initial value before any updates.

In [ ]:
horizontal = np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 160)
vertical = np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 160)
xx, yy = np.meshgrid(horizontal, vertical)
grid = np.column_stack([xx.ravel(), yy.ravel()])
probability_grid = model.predict_proba(scaler.transform(grid))[:, 1].reshape(xx.shape)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
filled = axes[0].contourf(
    xx, yy, probability_grid, levels=np.linspace(0, 1, 21), cmap="coolwarm", vmin=0, vmax=1
)
axes[0].contour(xx, yy, probability_grid, levels=[0.5], colors="black", linestyles="--")
axes[0].scatter(
    X_test[:, 0], X_test[:, 1], c=y_test, cmap="coolwarm", vmin=0, vmax=1, edgecolors="black", s=35
)
axes[0].set(xlabel="Feature 1", ylabel="Feature 2", title="Decision boundary: held-out data")
fig.colorbar(filled, ax=axes[0], label="P(class 1)")
axes[1].plot(np.arange(len(model.loss_history_)), model.loss_history_)
axes[1].set(xlabel="Gradient updates", ylabel="Regularized cross-entropy", title="Training loss")
fig.tight_layout()
plt.show()

## Stability at extreme predictions

The sigmoid avoids exponential overflow. Probability-based cross-entropy clips
endpoints before taking logarithms. Training uses the exact logit formulation so
confidently wrong predictions keep their true loss and gradient.

In [ ]:
np.testing.assert_array_equal(sigmoid([-1000.0, 0.0, 1000.0]), [0.0, 0.5, 1.0])
clipped_loss = binary_cross_entropy([1, 0], [0.0, 1.0])
assert np.isfinite(clipped_loss)
print(f"Finite clipped probability loss: {clipped_loss:.6f}")

# Run `python -m pytest tests/test_logistic_regression.py` to check the analytical
# gradient against central finite differences, including the intercept and L2.

Try different `alpha` values and learning rates. Compare precision and recall
when the sample balance changes. On separable data, unregularized coefficients
can keep growing; L2 prevents that. Next algorithm: k-means with k-means++ initialization.